## Machine Learning Assignment #1 - Image Classification

### Implement the missing parts to train an image classification model:
- configuration of the model.
- loss function for training the model.

### Extend the model to handle unknown classes:
- design a method that can identify inputs from unseen classes during inference
- the model should not only classify known classes but also detect and reject unknown samples
- there is no fixed or predefined answer
- include brief comments in the code to explain the approach used for unknown detection

In [ ]:
import torch
random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('current device: ',device)

current device:  cuda:0


### Importing libraries required for code execution.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
%matplotlib inline

import numpy as np
from PIL import Image

### Code for loading the provided dataset.

In [ ]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, pt_file):
        data = torch.load(pt_file)
        self.images = data['images']
        self.targets = data['labels']

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        image = self.images[idx].float() / 255.0
        target = self.targets[idx].long()
        return image, target

In [ ]:
def my_collate_fn(samples):
    images = []
    labels = []

    for data in samples:
        img, target = data
        images.append(img)
        labels.append(target)

    return torch.stack(images), torch.stack(labels)

### Data loading.
- "pt_file" referes to the location of the provided data file.
- the number of training samples is 50,000

In [ ]:
pt_file = os.path.join('./dataset/train.pt')

image_size = 128

train_dataset = CustomDataset(pt_file)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, collate_fn=my_collate_fn)

print(f'Length of train samples: {len(train_dataset)}')

Length of train samples: 50000


### Prepare the model; ResNet
- fill in the blank.
- ResNet-10 must be used without exception.
- If not implemented correctly, the training will not proceed properly.

In [ ]:
# Make the Basic Block of ResNet
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = # Fill in
        self.bn1 = # Fill in

        self.conv2 = # Fill in
        self.bn2 = # Fill in

        self.shortcut = nn.Sequential()
        if # Fill in
            self.shortcut = nn.Sequential(
                # Fill in
                nn.BatchNorm2d(planes))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = # Fill in
        out += # Fill in
        out = # Fill in
        return out


In [ ]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=100):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=2)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512, num_classes)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.gap(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNet10():
    return ResNet(BasicBlock, [1, 1, 1, 1])

In [ ]:
# Some code for saving and loading model.
# Use if you needed.
def save_model(model, save_name):
    path = './' + str(save_name) + '.pt'
    torch.save(model.state_dict(), path)

def load_model(init_model, load_name):
    path = './' + str(load_name) + '.pt'
    state_dict = torch.load(path, map_location='cpu')
    init_model.load_state_dict(state_dict)
    return init_model

#### Implementation of the loss function for training.
* You may implement and use various loss functions as needed.
    * e.g. cross entropy, weighted cross entropy, energy-based, label smoothing, etc, ...

In [ ]:
# customized loss function
def implemented_loss(output, target):
    # Fill in
    return # Fill in

### Train function

In [ ]:
def train(total_epoch, network, implemented_loss, optimizer, lr_schedule, train_loader, train_fn, device = 'cpu', save_name = 'save_name'):
    network = network.to(device)
    for epoch in range(total_epoch):
        train_fn(epoch, network, implemented_loss, optimizer, train_loader, device)
        lr_schedule.step()
        if ((epoch + 1) % 10 == 0) or epoch == total_epoch - 1:
            save_model(network, save_name)
            print('Model saved at epoch {} with name {} '.format(epoch + 1, save_name + '.pt'))

In [ ]:
def train_single_epoch(current_epoch, network, implemented_loss, optimizer, train_loader, device='cpu'):
    network.train()
    running_loss = 0.0
    loss_error = 0.0
    correct, total_sample = 0.0, 0.0
    for idx, (input, label) in enumerate(train_loader):
        input, label = input.to(device), label.to(device)
        optimizer.zero_grad()
        output = network(input)

        _, pred = torch.max(output.data, 1)
        correct += (pred == label.long()).sum().item()
        total_sample += label.size(0)

        loss = implemented_loss(output, label.view(-1).long())
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print('Epoch: {} | Training Accuracy: {:.2f} % | Loss: {:.2f}'.format(current_epoch, 100*correct/total_sample, running_loss/(idx+1)))

### Training begins!

In [ ]:
network = ResNet10()

total_epoch = 100
optimizer = optim.SGD(network.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
step_lr_scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[50, 75], gamma =0.1)

train_fn = train_single_epoch
train(total_epoch, network, implemented_loss, optimizer, step_lr_scheduler, train_loader, train_fn, device = device, save_name= f'name_studentID')

### Critical note regarding the testing process.
- the submitted models will be evaluated using the code that includes the test function below.
- make sure that the trained model runs inference properly with this code
- since the test data is not provided, verification can be done using the train data.
- failure to run will be treated as an error.
- Name the .pt file as: Name_StudentID.pt.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np


def roc_auc_score(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=np.float64)

    pos = (labels == 1)
    neg = (labels == 0)

    n_pos = pos.sum()
    n_neg = neg.sum()

    if n_pos == 0 or n_neg == 0:
        return 0.0

    order = np.argsort(scores)
    sorted_scores = scores[order]

    ranks = np.empty(len(scores), dtype=np.float64)

    i = 0
    while i < len(scores):
        j = i + 1
        while j < len(scores) and sorted_scores[j] == sorted_scores[i]:
            j += 1

        avg_rank = (i + 1 + j) / 2.0
        ranks[order[i:j]] = avg_rank
        i = j

    pos_ranks = ranks[pos].sum()
    auroc = (pos_ranks - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auroc)


def test(network, test_loader, device='cpu', load_name=None):
    if load_name is not None:
        network = load_model(network, load_name)

    network = network.to(device)
    network.eval()

    known_correct, known_total = 0, 0
    all_ood_scores = []
    all_ood_labels = []

    with torch.no_grad():
        for image, label in test_loader:
            image = image.to(device)
            label = label.to(device).view(-1).long()
            logits = network(image)
            prob = F.softmax(logits, dim=1)
            msp, pred_class = prob.max(dim=1)
            known_mask = (label != -1)

            if known_mask.any():
                known_correct += (pred_class[known_mask] == label[known_mask]).sum().item()
                known_total += known_mask.sum().item()

            ood_score = 1.0 - msp
            ood_label = (label == -1).long()
            all_ood_scores.append(ood_score.cpu().numpy())
            all_ood_labels.append(ood_label.cpu().numpy())

    known_acc = known_correct / known_total if known_total > 0 else 0.0
    all_ood_scores = np.concatenate(all_ood_scores)
    all_ood_labels = np.concatenate(all_ood_labels)
    auroc = roc_auc_score(all_ood_labels, all_ood_scores)
    final_score = 2 * known_acc * auroc / (known_acc + auroc)

    print('Known Accuracy: {:.4f} ({}/{})  Unknown Detection AUROC: {:.4f}  '
          'Final Score: {:.4f}'.format(known_acc, known_correct, known_total, auroc, final_score))
    return {'known_acc': known_acc, 'auroc': auroc,'final_score': final_score}


network = ResNet10()
test(network, train_loader, device = device, load_name='name_studentID')